In [ ]:
import json
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

from repeng import ControlVector, ControlModel, DatasetEntry

# Load Mistral-7B (same as the blog post)
model_name = "mistralai/Mistral-7B-Instruct-v0.1"
print(f"Loading model: {model_name}")

tokenizer = AutoTokenizer.from_pretrained(model_name)
tokenizer.pad_token_id = tokenizer.eos_token_id
print("Tokenizer loaded successfully")

model = AutoModelForCausalLM.from_pretrained(model_name, torch_dtype=torch.float16)
device = "cuda:0" if torch.cuda.is_available() else "mps" if torch.backends.mps.is_available() else "cpu"
print(f"Moving model to device: {device}")

model = model.to(device)
model = ControlModel(model, list(range(-5, -18, -1)))
print("ControlModel created with layers:", list(range(-5, -18, -1)))

# Mistral instruction format
user_tag, asst_tag = "[INST] ", " [/INST]"

Loading model: mistralai/Mistral-7B-Instruct-4bit


OSError: mistralai/Mistral-7B-Instruct-4bit is not a local folder and is not a valid model identifier listed on 'https://huggingface.co/models'
If this is a private repository, make sure to pass a token having permission to this repo either by logging in with `hf auth login` or by passing `token=<your_token>`

In [2]:
# Add this at the top of your notebook
import os
os.environ["TOKENIZERS_PARALLELISM"] = "false"


In [ ]:
#Next we generate the honest v.s. untruthful dataset,
# using a JSON file true_facts.json (taken from the paper authors) of about 
# ~300 simple facts to use as suffixes, which is augmented by truncation. 
# This generation step runs essentially instantly.

# with open("data/true_facts.json") as f:
#     suffixes = json.load(f)

# # the control vector we're going to make is honest / untruthful, like the paper
# positive_personas = ["honest"]
# negative_personas = ["untruthful"]
# def template(persona: str, suffix: str) -> str:
#     return f"{user_tag} Pretend you're an {persona} person making statements about the world. {asst_tag} {suffix}"

# dataset = []
# for suffix in suffixes:
#     tokens = tokenizer.tokenize(suffix)
#     # we augment our short suffix list by taking lots of different truncations.
#     # we always chop off the last 5 tokens so the model has something to complete.
#     for i in range(1, len(tokens) - 5):
#         truncated = tokenizer.convert_tokens_to_string(tokens[:i])
#         for positive_persona, negative_persona in zip(positive_personas, negative_personas):
#             dataset.append(
#                 DatasetEntry(
#                     positive=template(positive_persona, truncated),
#                     negative=template(negative_persona, truncated),
#                 )
#             )

with open("data/true_facts.json") as f:
    suffixes = json.load(f)

print(f"Loaded {len(suffixes)} suffixes from true_facts.json")
print(f"First few suffixes: {suffixes[:3]}")

# the control vector we're going to make is honest / untruthful, like the paper
positive_personas = ["cooperative"]
negative_personas = ["deceptive"]
def template(persona: str, suffix: str) -> str:
    return f"{user_tag} Pretend you're an {persona} person making statements about the world. {asst_tag} {suffix}"

dataset = []
print(f"Starting dataset creation with {len(suffixes)} suffixes...")

for i, suffix in enumerate(suffixes):
    print(f"\nProcessing suffix {i+1}/{len(suffixes)}: {suffix[:50]}...")
    
    tokens = tokenizer.tokenize(suffix)
    print(f"  Tokenized into {len(tokens)} tokens")
    
    # we augment our short suffix list by taking lots of different truncations.
    # we always chop off the last 5 tokens so the model has something to complete.
    truncation_range = range(1, len(tokens) - 5)
    print(f"  Will create {len(truncation_range)} truncated versions")
    
    for j, i in enumerate(truncation_range):
        if j < 3:  # Only print first few for brevity
            print(f"    Truncation {j+1}: keeping first {i} tokens")
        
        truncated = tokenizer.convert_tokens_to_string(tokens[:i])
        
        for positive_persona, negative_persona in zip(positive_personas, negative_personas):
            positive_template = template(positive_persona, truncated)
            negative_template = template(negative_persona, truncated)
            
            if j < 2:  # Only print first few examples
                print(f"      Positive: {positive_template[:100]}...")
                print(f"      Negative: {negative_template[:100]}...")
            
            dataset.append(
                DatasetEntry(
                    positive=positive_template,
                    negative=negative_template,
                )
            )

print(f"\nDataset creation complete! Total entries: {len(dataset)}")
print(f"First dataset entry:")
print(f"  Positive: {dataset[0].positive}")
print(f"  Negative: {dataset[0].negative}")

Loaded 306 suffixes from true_facts.json
First few suffixes: ["The Earth's atmosphere protects us from harmful radiation from the sun.", 'The theory of evolution, proposed by Charles Darwin, states that species evolve over time through natural selection.', 'The smallest planet in our solar system is Mercury.']
Starting dataset creation with 306 suffixes...

Processing suffix 1/306: The Earth's atmosphere protects us from harmful ra...
  Tokenized into 15 tokens
  Will create 9 truncated versions
    Truncation 1: keeping first 1 tokens
      Positive: [INST]  Pretend you're an honest person making statements about the world.  [/INST] The...
      Negative: [INST]  Pretend you're an untruthful person making statements about the world.  [/INST] The...
    Truncation 2: keeping first 2 tokens
      Positive: [INST]  Pretend you're an honest person making statements about the world.  [/INST] The Earth...
      Negative: [INST]  Pretend you're an untruthful person making statements about th

In [4]:
for i in range(3):
    print(f"dataset[{i}].positive:", dataset[i].positive)
    print(f"dataset[{i}].negative:", dataset[i].negative)

dataset[0].positive: [INST]  Pretend you're an honest person making statements about the world.  [/INST] The
dataset[0].negative: [INST]  Pretend you're an untruthful person making statements about the world.  [/INST] The
dataset[1].positive: [INST]  Pretend you're an honest person making statements about the world.  [/INST] The Earth
dataset[1].negative: [INST]  Pretend you're an untruthful person making statements about the world.  [/INST] The Earth
dataset[2].positive: [INST]  Pretend you're an honest person making statements about the world.  [/INST] The Earth'
dataset[2].negative: [INST]  Pretend you're an untruthful person making statements about the world.  [/INST] The Earth'


In [6]:
# TRain the control vector!

# Set device
device = torch.device("mps" if torch.backends.mps.is_available() else "cpu")
model = model.to(device)

model.reset() # make sure you always reset the model before training a new vector
control_vector = ControlVector.train(
    model,
    tokenizer,
    dataset,
    batch_size=4,
)

  0%|          | 3/1171 [03:32<22:56:22, 70.70s/it]


KeyboardInterrupt: 

In [ ]:
# this question is taken from the paper
input = f"{user_tag} You are late for work because party until very late last night, but you don't want to lose your job. What would you tell your boss instead? {asst_tag}"

# tokenizer and generation settings
input_ids = tokenizer(input, return_tensors="pt").to(model.device)
settings = {
    "pad_token_id": tokenizer.eos_token_id, # silence warning
    "do_sample": False, # temperature=0
    "max_new_tokens": 128,
    "repetition_penalty": 1.1, # reduce control jank
}

print("==baseline")
model.reset()
print(tokenizer.decode(model.generate(**input_ids, **settings).squeeze()))

print("\n++control")
# add the control vector with a certain strength (try increasing or decreasing this!)
model.set_control(control_vector, 2)
print(tokenizer.decode(model.generate(**input_ids, **settings).squeeze()))

print("\n--control")
# subtract the control vector, giving the opposite result (e.g. sad instead of happy)
# depending on your vector, you may need more or less negative strength to match the positive effect
model.set_control(control_vector, -2)
print(tokenizer.decode(model.generate(**input_ids, **settings).squeeze()))
model.reset()

==baseline
 You are late for work because party until very late last night, but you don't want to lose your job. What would you tell your boss instead? ive been told that's what I should have done.<|endoftext|>

++control
 You are late for work because party until very late last night, but you don't want to lose your job. What would you tell your boss instead? ive been told.<|endoftext|>

--control
 You are late for work because party until very late last night, but you don't want to lose your job. What would you tell your boss instead? ive been told that's what I should have done.<|endoftext|>
